# 時間発展回路の近似量子コンパイル
AQC-Tensor を使って、トロッター化された時間発展回路を量子ハードウェア上で効率的に実行できるよう圧縮する方法を学びます。

*使用量の目安: Heron プロセッサで 15 秒（注意: これはあくまで目安です。実際の実行時間は異なる場合があります。）*

## 学習成果

このチュートリアルを完了すると、次の内容が理解できるようになります。

* AQC-Tensor Qiskit アドオンを使って、深いトロッター回路を浅い ansatz 回路へ圧縮する方法
* トロッター回路からパラメーター付き ansatz を生成し、テンソルネットワーク（MPS）法でそのパラメーターを最適化する方法
* 圧縮された回路の忠実度を目標の時間発展と比較して評価し、量子ハードウェア上で実行する方法

## 前提知識

次のトピックについて、あらかじめ理解しておくことをお勧めします。

* [量子回路の基礎](/learning/courses/basics-of-quantum-information)
* [ハミルトニアンシミュレーションとトロッター化](/learning/courses/utility-scale-quantum-computing/quantum-simulation)
* [プリミティブの概要](/docs/guides/primitives)

## 背景

このチュートリアルでは、Qiskit を用いてテンソルネットワークによる**近似量子コンパイル**（AQC-Tensor）を実装し、量子回路の性能を高める方法を示します。AQC-Tensor は、シミュレーションの精度を保ちながら、深いトロッター回路をより浅くハードウェアに適した回路へ圧縮します。

### AQC-Tensor の仕組み

ハミルトニアン $H$ を全時間 $t$ にわたり $k$ 回のトロッターステップでシミュレートすることを考えます。完全なトロッター回路は次のように書けます。

$$
U_{\text{full}} = \left[U_{\text{Trotter}}(t/k)\right]^k
$$

素朴なアプローチでは、回路の深さを扱いやすく保つためにトロッターステップ数を少なくしますが、その代わりに大きなトロッター誤差が生じます。AQC-Tensor は、精度と深さを切り分けることでこのトレードオフを解消します。

1. **目標回路（高精度・深い）:** 同じ発展時間について、多数のステップ（たとえば $10k$）を用いたトロッター回路を構成します。この回路のトロッター誤差はずっと小さくなりますが、ハードウェアで実行するには深すぎます。行列積状態（MPS）として古典的にシミュレートするだけなので、深さは問題になりません。

2. **Ansatz 回路（浅い・パラメーター付き）:** 1 ステップのトロッター回路と同じ構造をもつ、パラメーター付き回路 $V(\theta)$ を定義します。$V(\theta_{\text{init}}) = U_{\text{Trotter}}(t/k)$ となるように初期化し、その後 $V(\theta)$ が高精度な目標状態をできる限り再現するように $\theta$ を反復的に最適化します。

その結果、深さは 1 トロッターステップ分のままでありながら、多数ステップ分の精度を達成する回路が得られ、近未来の量子ハードウェアでも実行可能になります。

### AQC-Tensor が有効な場面

AQC-Tensor が最も効果を発揮するのは、次のような場合です。

* **回路の深さがハードウェアのコヒーレンス時間を超える場合。** トロッターシミュレーションに必要なステップ数がデバイスで実行できる範囲を超えるとき、AQC-Tensor は時間発展をより浅い回路へ圧縮できます。
* **エンタングルメントが古典的に扱える範囲に収まる場合。** 時間発展した状態の全エンタングルメント量は、主に発展時間 $t$ に依存し、トロッターステップ数 $k$ には依存しません。つまり、$t$ が十分に短くボンド次元が扱える範囲に収まっていれば、$10k$ ステップの目標回路を MPS で表現する難しさは、$k$ ステップの場合と通常変わりません。
* **自然な ansatz が存在する場合。** ansatz はトロッター回路の構造を反映しているため、物理的な動機づけのある出発点と明確な初期パラメーターが得られ、任意の変分 ansatz で問題になりがちな収束の困難を避けられます。

このアプローチは、一般的な回路圧縮とは対照的です。任意のユニタリーをより少ないゲートで近似しようとするのではなく、AQC-Tensor はゲート構造を保ったままパラメーターを最適化してトロッター誤差を減らします。詳しくは [AQC-Tensor のドキュメント](https://qiskit.github.io/qiskit-addon-aqc-tensor/) を参照してください。

このチュートリアルでは、状態準備における AQC-Tensor のワークフロー全体を扱います。ハミルトニアンの定義、トロッター回路の生成、テンソルネットワーク最適化による圧縮、そして結果を IBM Quantum® ハードウェア上で実行するまでを順に見ていきます。

## 要件

このチュートリアルを始める前に、次のものがインストールされていることを確認してください。

* Qiskit SDK v2.0 以降（[可視化](/docs/api/qiskit/visualization) サポート付き）
* Qiskit Runtime v0.22 以降（`pip install qiskit-ibm-runtime`）
* AQC-Tensor Qiskit アドオン（`pip install 'qiskit-addon-aqc-tensor[aer,quimb-jax]'`）

## セットアップ

In [1]:
import numpy as np
import quimb.tensor
import datetime
import matplotlib.pyplot as plt

from scipy.linalg import expm
from scipy.optimize import OptimizeResult, minimize

from qiskit.quantum_info import SparsePauliOp, Pauli
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit import QuantumCircuit
from qiskit.synthesis import SuzukiTrotter

from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_aqc_tensor.ansatz_generation import (
    generate_ansatz_from_circuit,
)
from qiskit_addon_aqc_tensor.objective import MaximizeStateFidelity
from qiskit_addon_aqc_tensor.simulation.quimb import QuimbSimulator
from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
from qiskit_addon_aqc_tensor.simulation import compute_overlap

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.fake_provider import FakeKyiv

from rustworkx.visualization import graphviz_draw

## 小規模なシミュレーターでの例

このセクションでは、10 サイト系を用いて AQC-Tensor のワークフローを段階的に説明します。ここでは、スピン間相互作用や磁気的性質の研究で広く用いられているモデルである、10 サイトの XXZ スピン鎖のダイナミクスをシミュレートします。

ハミルトニアンは次のとおりです。

$$
\hat{\mathcal{H}}_{XXZ} = \sum_{i=1}^{L-1} J_{i,(i+1)}\left(X_i X_{(i+1)}+Y_i Y_{(i+1)}+ 2\cdot Z_i Z_{(i+1)} \right) \, ,
$$

ここで $J_{i,(i+1)}$ は辺 $(i, i+1)$ に対するランダムな係数で、$L=10$ です。

### ステップ 1: 古典的な入力を量子問題へマッピングする

このステップでは、次のことを行います。

1. ハミルトニアン、観測量、初期状態を定義する。
2. 後の比較のために、期待値の厳密解を古典的に計算する。
3. 高精度なトロッター回路（AQC の目標）を生成し、AQC-Tensor で浅い ansatz へ圧縮する。

#### ハミルトニアン、観測量、初期状態を設定する

In [2]:
# L は 1 次元スピン鎖のサイト数
L = 10

# カップリングマップを生成する
edge_list = [(i - 1, i) for i in range(1, L)]
even_edges = edge_list[::2]
odd_edges = edge_list[1::2]
coupling_map = CouplingMap(edge_list)

# XXZ ハミルトニアン用のランダムな係数を生成する
np.random.seed(0)
Js = np.random.rand(L - 1) + 0.5 * np.ones(L - 1)
hamiltonian = SparsePauliOp(Pauli("I" * L))
for i, edge in enumerate(even_edges + odd_edges):
    hamiltonian += SparsePauliOp.from_sparse_list(
        [
            ("XX", (edge), Js[i] / 2),
            ("YY", (edge), Js[i] / 2),
            ("ZZ", (edge), Js[i]),
        ],
        num_qubits=L,
    )

# 中央の 2 量子ビット間の ZZ 観測量を生成する
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

# 初期状態としてネール状態 |1010101010⟩ を生成する
initial_state_circuit = QuantumCircuit(L)
for i in range(L):
    if i % 2:
        initial_state_circuit.x(i)

print("Hamiltonian:", hamiltonian)
print("Observable:", observable)
graphviz_draw(coupling_map.graph, method="circo")

Hamiltonian: SparsePauliOp(['IIIIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII'],
              coeffs=[1.        +0.j, 0.52440675+0.j, 0.52440675+0.j, 1.0488135 +0.j,
 0.60759468+0.j, 0.60759468+0.j, 1.21518937+0.j, 0.55138169+0.j,
 0.55138169+0.j, 1.10276338+0.j, 0.52244159+0.j, 0.52244159+0.j,
 1.04488318+0.j, 0.4618274 +0.j, 0.4618274 +0.j, 0.9236548 +0.j,
 0.57294706+0.j, 0.57294706+0.j, 1.14589411+0.j, 0.46879361+0.j,
 0.46879361+0.j, 0.93758721+0.j, 0.6958865 +0.j, 0.6958865 +0.j,
 1.391773  +0.j, 0.73183138+0.j, 0.73183138+0.j, 1.46366276+0.j])
Observable: SparsePauliOp(['IIIIZZIIII'],
              coeffs=[1.+0.j])


<Image src="/docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/527dbada-1.avif" alt="Output of the previous code cell" />

#### 期待値の厳密解を計算する

このサイズの系であれば、行列指数関数を用いて時間発展後の期待値を直接厳密に計算できます。これが AQC 回路の精度を評価する際の基準（真の値）になります。

In [3]:
aqc_evolution_time = 0.2

# ベースラインの各トロッターステップは dt = aqc_evolution_time / 3 に対応する
# 後続の（非圧縮の）ステップはさらに 1 dt 分に対応する
subsequent_evolution_time = aqc_evolution_time / 3
total_evolution_time = aqc_evolution_time + subsequent_evolution_time

# 行列指数関数により期待値の厳密解を計算する
H_matrix = hamiltonian.to_matrix()
U_exact = expm(-1j * H_matrix * total_evolution_time)

# 初期状態ベクトル（ネール状態）を構築する
initial_state_vec = np.zeros(2**L)
state_idx = sum(2**i for i in range(L) if i % 2)
initial_state_vec[state_idx] = 1.0

# 時間発展させて期待値を計算する
evolved_state = U_exact @ initial_state_vec
obs_matrix = observable.to_matrix()
exact_expval = (evolved_state.conj() @ obs_matrix @ evolved_state).real

print(f"AQC evolution time: {aqc_evolution_time}")
print(f"Subsequent evolution time: {subsequent_evolution_time:.6f}")
print(f"Total evolution time: {total_evolution_time:.6f}")
print(f"Exact expectation value: {exact_expval:.6f}")

AQC evolution time: 0.2
Subsequent evolution time: 0.066667
Total evolution time: 0.266667
Exact expectation value: -0.700899


#### AQC の目標回路を生成する

次に、AQC の目標となるトロッター回路を構成します。この回路は高精度を得るために多数のトロッターステップ（32）を使います。ハードウェアで実行するのではなく MPS として古典的にシミュレートするだけなので、深さが大きくても問題ありません。

In [4]:
aqc_target_num_trotter_steps = 32

aqc_target_circuit = initial_state_circuit.copy()
aqc_target_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
        time=aqc_evolution_time,
    ),
    inplace=True,
)

#### Ansatz、初期パラメーター、後続回路、およびベースライン回路を生成する

続いて、AQC の目標と同じ発展時間をもちながら、トロッターステップ数がはるかに少ない（1 ステップだけの）「良い」回路を構成します。この回路を `generate_ansatz_from_circuit` に渡すと、次のものが返されます。

1. 同じ 2 量子ビット接続をもつ、一般的でパラメーター付きの **ansatz** 回路。
2. ansatz に代入すると入力回路を再現する **初期パラメーター**。

さらに、次のものも構成します。

* AQC で最適化した部分の後に（圧縮せずに）付け加える、1 トロッターステップの **後続回路**。これは [AQC-Tensor の初期状態チュートリアル](https://qiskit.github.io/qiskit-addon-aqc-tensor/tutorials/01_initial_state_aqc.html) のアプローチに従っています。
* 全発展時間（`aqc_evolution_time + subsequent_evolution_time`）にわたり 4 トロッターステップを用いた **ベースラインのトロッター回路**。これは比較対象であり、AQC を使わずにハードウェアで実行した場合に相当します。AQC ansatz（圧縮 3 ステップ + 非圧縮 1 ステップ）は、より浅い深さでより高い精度を達成します。

In [ ]:
aqc_ansatz_num_trotter_steps = 1

aqc_good_circuit = initial_state_circuit.copy()
aqc_good_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=aqc_ansatz_num_trotter_steps),
        time=aqc_evolution_time,
    ),
    inplace=True,
)

aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(
    aqc_good_circuit
)

# 後続回路: AQC の後に付け加える、圧縮していない 1 トロッターステップ
subsequent_num_trotter_steps = 1
subsequent_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=subsequent_num_trotter_steps),
    time=subsequent_evolution_time,
)

# ベースラインのトロッター回路: 全発展時間にわたる 4 トロッターステップ（AQC なし）
baseline_num_trotter_steps = 4
baseline_circuit = initial_state_circuit.copy()
baseline_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=baseline_num_trotter_steps),
        time=total_evolution_time,
    ),
    inplace=True,
)

print(
    f"Target circuit:      depth {aqc_target_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"Baseline circuit:    depth {baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)} ({baseline_num_trotter_steps} Trotter steps, time={total_evolution_time:.4f})"
)
print(
    f"Subsequent circuit:  depth {subsequent_circuit.depth(lambda x: x.operation.num_qubits == 2)} ({subsequent_num_trotter_steps} Trotter step, time={subsequent_evolution_time:.4f})"
)
print(
    f"Ansatz circuit:      depth {aqc_ansatz.depth(lambda x: x.operation.num_qubits == 2)}, with {len(aqc_initial_parameters)} parameters"
)
aqc_ansatz.draw("mpl", fold=-1)

Target circuit:      depth 384
Baseline circuit:    depth 48 (4 Trotter steps, time=0.2667)
Subsequent circuit:  depth 12 (1 Trotter step, time=0.0667)
Ansatz circuit:      depth 3, with 156 parameters


<Image src="/docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/78f2665e-1.avif" alt="Output of the previous code cell" />

#### テンソルネットワークシミュレーションを設定し、目標 MPS を構築する

ここでは行列積状態（MPS）の回路シミュレーターとして [quimb](https://github.com/jcmgray/quimb) を使い、勾配ベースの最適化のための自動微分は JAX が担います。続いて目標状態の MPS 表現を構築し、初期 ansatz と目標との間の初期忠実度を評価します。今回の問題例は比較的小規模なため、初期忠実度は最初からかなり高い値になります。

In [6]:
simulator_settings = QuimbSimulator(
    quimb.tensor.CircuitMPS, autodiff_backend="jax"
)

aqc_target_mps = tensornetwork_from_circuit(
    aqc_target_circuit, simulator_settings
)
print("Target MPS maximum bond dimension:", aqc_target_mps.psi.max_bond())

good_mps = tensornetwork_from_circuit(aqc_good_circuit, simulator_settings)
starting_fidelity = abs(compute_overlap(good_mps, aqc_target_mps)) ** 2
print(f"Starting fidelity: {starting_fidelity:.6f}")

Target MPS maximum bond dimension: 5
Starting fidelity: 0.998246


#### Ansatz のパラメーターを最適化する

L-BFGS-B オプティマイザーを用いて `MaximizeStateFidelity` コスト関数を最小化します。オプティマイザーは ansatz のパラメーターを反復的に調整し、ansatz 回路と目標 MPS との忠実度を最大化します。

In [7]:
aqc_stopping_fidelity = 1
aqc_max_iterations = 500

stopping_point = 1.0 - aqc_stopping_fidelity
objective = MaximizeStateFidelity(
    aqc_target_mps, aqc_ansatz, simulator_settings
)


def callback(intermediate_result: OptimizeResult):
    fidelity = 1 - intermediate_result.fun
    print(
        f"{datetime.datetime.now()} Intermediate result: Fidelity {fidelity:.8f}"
    )
    if intermediate_result.fun < stopping_point:
        raise StopIteration


result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": aqc_max_iterations},
    callback=callback,
)
if result.status not in (0, 1, 99):
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )

print(f"Done after {result.nit} iterations.")
aqc_final_parameters = result.x

2026-05-18 13:14:49.731596 Intermediate result: Fidelity 0.99952882
2026-05-18 13:14:49.734425 Intermediate result: Fidelity 0.99958531
2026-05-18 13:14:49.737101 Intermediate result: Fidelity 0.99960093
2026-05-18 13:14:49.739813 Intermediate result: Fidelity 0.99961046
2026-05-18 13:14:49.742969 Intermediate result: Fidelity 0.99962560
2026-05-18 13:14:49.745916 Intermediate result: Fidelity 0.99964395
2026-05-18 13:14:49.748615 Intermediate result: Fidelity 0.99968150
2026-05-18 13:14:49.753684 Intermediate result: Fidelity 0.99970569
2026-05-18 13:14:49.756208 Intermediate result: Fidelity 0.99973788
2026-05-18 13:14:49.759067 Intermediate result: Fidelity 0.99975385
2026-05-18 13:14:49.762321 Intermediate result: Fidelity 0.99976458
2026-05-18 13:14:49.765526 Intermediate result: Fidelity 0.99977661
2026-05-18 13:14:49.768496 Intermediate result: Fidelity 0.99978663
2026-05-18 13:14:49.771278 Intermediate result: Fidelity 0.99980236
2026-05-18 13:14:49.773735 Intermediate result: 

#### 最終的な AQC 回路を組み立てる

最適化されたパラメーターが得られたので、それを ansatz に代入し、その後に（圧縮していない）後続のトロッターステップを付け加えます。得られる回路の深さは、圧縮された 1 トロッターステップ分と非圧縮 1 ステップ分の合計ですが、圧縮された部分は 32 トロッターステップ分の精度を近似しています。

In [8]:
aqc_final_circuit = aqc_ansatz.assign_parameters(aqc_final_parameters)
aqc_final_circuit.compose(subsequent_circuit, inplace=True)
aqc_final_circuit.draw("mpl", fold=-1)

<Image src="/docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/e09e40de-0.avif" alt="Output of the previous code cell" />

### ステップ 2: 量子ハードウェア実行に向けて問題を最適化する

この小規模な例では、ハードウェア実行をローカルで模擬するために擬似バックエンド（`FakeKyiv`）を使います。AQC で最適化した回路（`aqc_final_circuit`）と、ベースラインのトロッター回路（`baseline_circuit`、全発展時間にわたる 4 トロッターステップ、AQC なし）の両方を、バックエンドの命令セットアーキテクチャ（ISA）へトランスパイルします。回路の深さをさらに削減するため `optimization_level=3` を指定します。

In [9]:
backend = FakeKyiv()

pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=3
)

# AQC で最適化した回路（圧縮部分 + 後続ステップ）をトランスパイルする
isa_circuit = pass_manager.run(aqc_final_circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(
    "AQC circuit depth:",
    isa_circuit.depth(lambda x: x.operation.num_qubits == 2),
)

# ベースラインのトロッター回路（AQC 最適化なし）をトランスパイルする
isa_baseline_circuit = pass_manager.run(baseline_circuit)
isa_baseline_observable = observable.apply_layout(isa_baseline_circuit.layout)
print(
    "Baseline Trotter circuit depth:",
    isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2),
)

AQC circuit depth: 15
Baseline Trotter circuit depth: 27


### ステップ 3: Qiskit プリミティブを使って実行する

擬似バックエンド上で [`EstimatorV2`](/docs/api/qiskit-ibm-runtime/estimator-v2) プリミティブを使い、AQC で最適化した回路とベースラインのトロッター回路の両方を実行し、それぞれについて ZZ 観測量を測定します。

In [10]:
estimator = Estimator(backend)

# 両方の回路を実行する
aqc_result = estimator.run([(isa_circuit, isa_observable)]).result()
baseline_result = estimator.run(
    [(isa_baseline_circuit, isa_baseline_observable)]
).result()

### ステップ 4: 後処理を行い、望ましい古典的形式で結果を返す

両方の実行から期待値を取り出し、厳密解と比較します。ベースラインのトロッター回路は、同じ回路の深さで AQC を使わなかった場合に得られる結果を示し、AQC 回路はテンソルネットワーク最適化による改善を示します。

In [11]:
aqc_expval = aqc_result[0].data.evs.tolist()
baseline_expval = baseline_result[0].data.evs.tolist()

print(f"Exact:              {exact_expval:.4f}")
print(
    f"Baseline Trotter:   {baseline_expval:.4f}, |\u0394| = {np.abs(exact_expval - baseline_expval):.4f}  (depth {isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)}, {baseline_num_trotter_steps} steps)"
)
print(
    f"AQC (3+1):          {aqc_expval:.4f}, |\u0394| = {np.abs(exact_expval - aqc_expval):.4f}  (depth {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}, compressed+subsequent)"
)

Exact:              -0.7009
Baseline Trotter:   -0.5400, |Δ| = 0.1609  (depth 27, 4 steps)
AQC (3+1):          -0.5728, |Δ| = 0.1281  (depth 15, compressed+subsequent)


In [12]:
plt.style.use("seaborn-v0_8")

labels = [
    f"Baseline Trotter\n({baseline_num_trotter_steps} steps, depth {isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)})",
    f"AQC (3+1)\n(depth {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)})",
]
values = [baseline_expval, aqc_expval]
colors = ["tab:orange", "tab:blue"]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values, color=colors, width=0.5)
plt.axhline(
    y=exact_expval,
    color="tab:green",
    linestyle="--",
    linewidth=2,
    label=f"Exact ({exact_expval:.4f})",
)
plt.ylabel("Expected Value")
plt.title(
    "AQC-Tensor (3 compressed + 1 uncompressed) vs Baseline Trotter (10-site XXZ)"
)
plt.legend()
for bar in bars:
    y_val = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        y_val,
        f"{y_val:.4f}",
        ha="center",
        va="bottom" if y_val >= 0 else "top",
    )
plt.axhline(y=0, color="black", linewidth=0.3)
plt.tight_layout()
plt.show()

<Image src="/docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/77c39ba8-0.avif" alt="Output of the previous code cell" />

## 大規模なハードウェアでの例

ここからは 50 サイトの XXZ モデルへスケールアップし、より現実的な問題サイズで AQC-Tensor を実演します。ワークフローは小規模な例と同じで、3 トロッターステップを AQC で圧縮し、非圧縮の 1 ステップを付け加えます。

このサイズの系では行列指数関数の計算は現実的ではない（次元が $2^{50}$）ため、参照となる期待値は、全時間にわたって発展させた高精度な MPS から直接計算します。

### ステップ 1〜4 をまとめて実行

In [13]:
# -------------------------ステップ 1-------------------------

# 50 サイトのスピン鎖を定義する
L = 50
edge_list = [(i - 1, i) for i in range(1, L)]
even_edges = edge_list[::2]
odd_edges = edge_list[1::2]
coupling_map = CouplingMap(edge_list)

# ランダムな XXZ ハミルトニアン
np.random.seed(0)
Js = np.random.rand(L - 1) + 0.5 * np.ones(L - 1)
hamiltonian = SparsePauliOp(Pauli("I" * L))
for i, edge in enumerate(even_edges + odd_edges):
    hamiltonian += SparsePauliOp.from_sparse_list(
        [
            ("XX", (edge), Js[i] / 2),
            ("YY", (edge), Js[i] / 2),
            ("ZZ", (edge), Js[i]),
        ],
        num_qubits=L,
    )

observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

# 初期状態のネール状態
initial_state_circuit = QuantumCircuit(L)
for i in range(L):
    if i % 2:
        initial_state_circuit.x(i)

# 時間のパラメーター
aqc_evolution_time = 0.2
subsequent_evolution_time = aqc_evolution_time / 3
total_evolution_time = aqc_evolution_time + subsequent_evolution_time

# AQC の目標回路（高精度。AQC 部分は 32 トロッターステップ）
aqc_target_num_trotter_steps = 32

aqc_target_circuit = initial_state_circuit.copy()
aqc_target_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
        time=aqc_evolution_time,
    ),
    inplace=True,
)

# 1 ステップのトロッター回路から ansatz を生成する
aqc_good_circuit = initial_state_circuit.copy()
aqc_good_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=1),
        time=aqc_evolution_time,
    ),
    inplace=True,
)

aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(
    aqc_good_circuit
)

# 後続回路: 圧縮していない 1 トロッターステップ
subsequent_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=1),
    time=subsequent_evolution_time,
)

# ベースラインのトロッター回路: 全発展時間にわたる 4 トロッターステップ（AQC なし）
baseline_num_trotter_steps = 4
baseline_circuit = initial_state_circuit.copy()
baseline_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=baseline_num_trotter_steps),
        time=total_evolution_time,
    ),
    inplace=True,
)
print(
    f"Target circuit:  depth {aqc_target_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"Ansatz circuit:  depth {aqc_ansatz.depth(lambda x: x.operation.num_qubits == 2)}, with {len(aqc_initial_parameters)} parameters"
)
print(
    f"Subsequent circuit: depth {subsequent_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"Baseline circuit:   depth {baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)} ({baseline_num_trotter_steps} steps, time={total_evolution_time:.4f})"
)

# 目標 MPS を構築し、参照となる期待値を計算する
simulator_settings = QuimbSimulator(
    quimb.tensor.CircuitMPS, autodiff_backend="jax"
)
aqc_target_mps = tensornetwork_from_circuit(
    aqc_target_circuit, simulator_settings
)
print("Target MPS maximum bond dimension:", aqc_target_mps.psi.max_bond())

# 参照期待値には全体の時間発展（AQC + 後続）が必要
# MPS の参照用に高精度な全体回路を構築する
full_target_circuit = initial_state_circuit.copy()
full_target_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
        time=total_evolution_time,
    ),
    inplace=True,
)
full_target_mps = tensornetwork_from_circuit(
    full_target_circuit, simulator_settings
)
exact_expval = full_target_mps.local_expectation(
    quimb.pauli("Z") & quimb.pauli("Z"), (L // 2 - 1, L // 2)
).real.item()
print(f"Reference expectation value (from MPS): {exact_expval:.6f}")

# ansatz のパラメーターを最適化する
objective = MaximizeStateFidelity(
    aqc_target_mps, aqc_ansatz, simulator_settings
)


def callback(intermediate_result: OptimizeResult):
    fidelity = 1 - intermediate_result.fun
    print(
        f"{datetime.datetime.now()} Intermediate result: Fidelity {fidelity:.8f}"
    )


result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": 500},
    callback=callback,
)
if result.status not in (0, 1, 99):
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )
print(f"Done after {result.nit} iterations.")

# 最終的な AQC 回路を組み立てる: 最適化した ansatz + 後続のトロッターステップ
aqc_final_circuit = aqc_ansatz.assign_parameters(result.x)
aqc_final_circuit.compose(subsequent_circuit, inplace=True)

# -------------------------ステップ 2-------------------------

service = QiskitRuntimeService()
backend = service.least_busy(min_num_qubits=127)
print(backend)

pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=3
)
isa_circuit = pass_manager.run(aqc_final_circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(
    "AQC circuit depth:",
    isa_circuit.depth(lambda x: x.operation.num_qubits == 2),
)

# ベースラインのトロッター回路（4 トロッターステップ、AQC なし）もトランスパイルする
isa_baseline_circuit = pass_manager.run(baseline_circuit)
isa_baseline_observable = observable.apply_layout(isa_baseline_circuit.layout)
print(
    "Baseline Trotter circuit depth:",
    isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2),
)

# -------------------------ステップ 3-------------------------

# 両方の回路を 1 つのジョブとして投入する
estimator = Estimator(backend)
estimator.options.environment.job_tags = ["TUT_AQCTE"]

job = estimator.run(
    [
        (isa_circuit, isa_observable),
        (isa_baseline_circuit, isa_baseline_observable),
    ]
)
print("Job ID:", job.job_id())

Target circuit:  depth 385
Ansatz circuit:  depth 7, with 816 parameters
Subsequent circuit: depth 12
Baseline circuit:   depth 49 (4 steps, time=0.2667)
Target MPS maximum bond dimension: 5
Reference expectation value (from MPS): -0.738669
2026-05-18 13:02:11.219150 Intermediate result: Fidelity 0.99795732
2026-05-18 13:02:11.232256 Intermediate result: Fidelity 0.99822481
2026-05-18 13:02:11.245160 Intermediate result: Fidelity 0.99829520
2026-05-18 13:02:11.257765 Intermediate result: Fidelity 0.99832379
2026-05-18 13:02:11.270280 Intermediate result: Fidelity 0.99836416
2026-05-18 13:02:11.284116 Intermediate result: Fidelity 0.99840073
2026-05-18 13:02:11.296856 Intermediate result: Fidelity 0.99846863
2026-05-18 13:02:11.309602 Intermediate result: Fidelity 0.99865244
2026-05-18 13:02:11.322012 Intermediate result: Fidelity 0.99872665
2026-05-18 13:02:11.334195 Intermediate result: Fidelity 0.99892335
2026-05-18 13:02:11.346570 Intermediate result: Fidelity 0.99901045
2026-05-18 

In [15]:
# -------------------------ステップ 4-------------------------

hw_results = job.result()
aqc_expval = hw_results[0].data.evs.tolist()
baseline_expval = hw_results[1].data.evs.tolist()

print(f"Exact (MPS):        {exact_expval:.4f}")
print(
    f"Baseline Trotter:   {baseline_expval:.4f}, |\u0394| = {np.abs(exact_expval - baseline_expval):.4f}"
)
print(
    f"AQC (3+1):          {aqc_expval:.4f}, |\u0394| = {np.abs(exact_expval - aqc_expval):.4f}"
)

labels = [
    f"Baseline Trotter\n({baseline_num_trotter_steps} steps, depth {isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)})",
    f"AQC (3+1)\n(depth {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)})",
]
values = [baseline_expval, aqc_expval]
colors = ["tab:orange", "tab:blue"]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values, color=colors, width=0.5)
plt.axhline(
    y=exact_expval,
    color="tab:green",
    linestyle="--",
    linewidth=2,
    label=f"Exact ({exact_expval:.4f})",
)
plt.ylabel("Expected Value")
plt.title(
    "AQC-Tensor (3 compressed + 1 uncompressed) vs Baseline Trotter (50-site XXZ)"
)
plt.legend()
for bar in bars:
    y_val = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        y_val,
        f"{y_val:.4f}",
        ha="center",
        va="bottom" if y_val >= 0 else "top",
    )
plt.axhline(y=0, color="black", linewidth=0.3)
plt.tight_layout()
plt.show()

Exact (MPS):        -0.7387
Baseline Trotter:   -0.5955, |Δ| = 0.1432
AQC (3+1):          -0.6734, |Δ| = 0.0653


<Image src="/docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/a4dc23fd-494e-46cb-a8f5-d1cd444b96f4-1.avif" alt="Output of the previous code cell" />

## 次のステップ

<Admonition type="tip" title="おすすめ">
  この内容に興味を持たれた方には、次の資料もお勧めします。

  * [AQC-Tensor アドオンのドキュメント](https://qiskit.github.io/qiskit-addon-aqc-tensor/) — 準備した状態ではなく目標のユニタリー演算子を近似するようにパラメーター付き回路を最適化する、関連手法の **unitary AQC** についても解説されています
  * [誤差抑制・誤差低減の手法](/docs/guides/error-mitigation-and-suppression-techniques)
  * [誤差低減手法を組み合わせる](/docs/tutorials/combine-error-mitigation-techniques)
</Admonition>

© IBM Corp., 2017-2026